# HW_03 — Embeddings (تمرین سوم: embeddings)

**درس:** Generative AI — دانشکار

**یادداشت اجرا:** طبق نسخه‌ی اصلی تمرین، بخش اول باید در محیط Google Colab انجام شود. در این نسخه، **کل تمرین به صورت لوکال روی سیستم شخصی** اجرا شده است:
- برای بخش ۱، به‌جای نصب Ollama در Colab، از سرور Ollama که از قبل روی سیستم نصب و در حال اجراست استفاده شده و مدل `llama3.1:8b` (که از قبل دانلود شده بود) سرو شده است.
- برای بخش ۲، مدل embedding فارسی `HooshvareLab/bert-base-parsbert-uncased` به صورت لوکال دانلود و اجرا شده است.
- برای بخش ۳، سه مقاله‌ی واقعی از ویکی‌پدیای فارسی (هوش مصنوعی، یادگیری ماشین، پردازش زبان‌های طبیعی) دانلود و به عنوان سه منبع داده استفاده شده‌اند.

این notebook شامل هر ۳ بخش تمرین است که هرکدام با markdown مجزا مشخص شده‌اند:
1. راه‌اندازی Ollama و اتصال با LangChain
2. استفاده از مدل embedding به‌صورت لوکال + ساخت vectorstore با FAISS
3. فیلتر کردن جستجوی برداری بر اساس متادیتا

---

## بخش ۱: راه‌اندازی Ollama و اتصال با LangChain

در این بخش، مدل زبانی متن‌باز `llama3.1` را از طریق Ollama سرو کرده و با استفاده از کتابخانه‌ی `langchain-ollama` یک instance از مدل ایجاد کرده و پرامپ‌های مختلف (فارسی و انگلیسی) را به آن ارسال می‌کنیم.

> **تفاوت با دستورالعمل اصلی:** دستورات colab (`curl -fsSL https://ollama.com/install.sh | sh`, `!nohup ollama serve &`, `!ollama run llama3.1`) جایگزین شده‌اند با یک سرور Ollama که از قبل به صورت local service روی مک در حال اجراست (`ollama serve`) و مدل `llama3.1:8b` که با `ollama pull llama3.1:8b` دانلود شده است.

### مرحله ۱: بررسی در دسترس بودن سرور Ollama و مدل

In [1]:
import requests

resp = requests.get("http://localhost:11434/api/tags")
resp.raise_for_status()
models = [m["name"] for m in resp.json()["models"]]

print("Ollama در حال اجراست. مدل‌های موجود روی سیستم:")
for m in models:
    print(" -", m)

assert "llama3.1:8b" in models, "مدل llama3.1:8b یافت نشد؛ ابتدا با 'ollama pull llama3.1:8b' آن را دانلود کنید."
print("\nمدل llama3.1:8b آماده‌ی استفاده است.")

Ollama در حال اجراست. مدل‌های موجود روی سیستم:
 - nomic-embed-text:latest
 - llama3.1:8b

مدل llama3.1:8b آماده‌ی استفاده است.


### مرحله ۲: اتصال به سرور Ollama با استفاده از LangChain (`ChatOllama`)

In [2]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.1:8b",
    base_url="http://localhost:11434",
    temperature=0.3,
)

print("مدل زبانی با موفقیت بارگذاری شد:", llm.model)

/Users/pouyanmb/Desktop/AI_ENGINEERING_BOOTCAMP_DANESHKAR/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


مدل زبانی با موفقیت بارگذاری شد: llama3.1:8b


### تست عملکرد مدل با پرامپت فارسی

In [3]:
response_fa = llm.invoke(
    "در یک پاراگراف کوتاه توضیح بده embedding در پردازش زبان طبیعی به چه معناست و چه کاربردی دارد؟"
)
print(response_fa.content)

در پردازش زبان طبیعی، Embedding (به فارسی: گنجاندن) یک تکنیک است که برای تبدیل کلمات یا توابع واژگانی به ماتریکس‌های با ابعاد بالا استفاده می‌شود. این تکنیک به ما کمک می‌کند تا معنای کلمات را در یک فضای numbrique نمایش دهیم و از آن‌ها برای وظایف مختلف پردازش زبان طبیعی مانند طبقه‌بندی، ترجمه ماشین و فهم متن استفاده کنیم.


### تست عملکرد مدل با پرامپت انگلیسی

In [4]:
response_en = llm.invoke(
    "In one short paragraph, explain what a vector database is used for."
)
print(response_en.content)

A vector database is a type of database designed to efficiently store and query high-dimensional vectors, such as those used in computer vision and natural language processing applications. It's typically used for tasks like image or video similarity search, facial recognition, text classification, and recommendation systems, where the ability to quickly compare and retrieve similar vectors is crucial.


### تست با پیام سیستمی (System + Human message)

In [5]:
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage(content="تو یک دستیار متخصص یادگیری ماشین هستی و باید مختصر و دقیق پاسخ بدهی."),
    HumanMessage(content="تفاوت اصلی بین embedding های محلی (local) مثل ParsBERT و embedding های ارائه‌شده از طریق API مثل OpenAI چیست؟"),
]

response = llm.invoke(messages)
print(response.content)

پاسخ: تفاوت اصلی بین این دو نوع Embedding در نحوه آموزش و استفاده از مدل یادگیری ماشین است.

- Embeddings محلی مانند ParsBERT به صورت داخلی در یک محیط خاص آموزش داده می‌شوند. این مدل‌ها برای پردازش زبان طبیعی (NLP) طراحی شده‌اند و می‌توانند بر روی داده‌های خاصی که در اختیار دارند، عملکرد خوبی نشان دهند.
- در مقابل، API های مانند OpenAI از مدل های پیش‌ساخته استفاده می‌کنند که به صورت گسترده روی مجموعه‌های داده بزرگ آموزش داده شده‌اند. این مدل ها برای پردازش متن‌های مختلف طراحی شده اند و می‌توانند بر روی داده‌های جدید عملکرد خوبی نشان دهند.

در کل، Embeddings محلی مناسب هستند اگر نیاز دارید یک مدل خاص برای یک محیط یا کاربرد خاص باشد، در حالی که API های مانند OpenAI مناسب هستند اگر نیاز دارید یک مدل پیش‌ساخته با عملکرد بالا برای پردازش متن‌های مختلف داشته باشید.


---

## بخش ۲: استفاده از مدل‌های embedding به‌صورت لوکال

در این بخش، یک vectorstore با استفاده از FAISS و یک مدل embedding که قابلیت اجرا به‌صورت لوکال دارد بر روی متون فارسی پیاده‌سازی می‌کنیم.

### بخش اول: دانلود مدل embedding (ParsBERT)

In [6]:
%pip install -q langchain langchain-huggingface langchain_community faiss-cpu

Note: you may need to restart the kernel to use updated packages.


In [7]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

model_name = "HooshvareLab/bert-base-parsbert-uncased"
model_kwargs = {'device': 'cpu'}
encode_kwargs = {'normalize_embeddings': False}

hf_embedding = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
)

print("مدل embedding فارسی (ParsBERT) با موفقیت بارگذاری شد.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 35265.61it/s]


[transformers] BertModel LOAD REPORT from: HooshvareLab/bert-base-parsbert-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


مدل embedding فارسی (ParsBERT) با موفقیت بارگذاری شد.


حال برای تست عملکرد مدل، کد زیر را اجرا می‌کنیم:

In [8]:
text = (
    "ما در هوشواره معتقدیم با انتقال صحیح دانش و آگاهی، همه افراد "
    "می‌توانند از ابزارهای هوشمند استفاده کنند. شعار ما هوش مصنوعی برای همه است."
)

embed = hf_embedding.embed_query(text)
print("طول بردار embedding:", len(embed))
print("۵ مقدار اول بردار:", embed[:5])

طول بردار embedding: 768
۵ مقدار اول بردار: [-0.3744332492351532, -0.41980355978012085, -0.27720150351524353, 1.9151549339294434, -0.07415932416915894]


### بخش دوم: ساخت vectorstore

یک متن فارسی (بخشی از مقاله‌ی «هوش مصنوعی» ویکی‌پدیای فارسی) را به chunk‌های مناسب تبدیل کرده و با استفاده از FAISS یک vectorstore با قابلیت جستجوی معنایی می‌سازیم.

In [9]:
with open("data/ai.txt", encoding="utf-8") as f:
    ai_text = f.read()

sample_text = ai_text[:2000]  # بخشی از مقاله برای دموی ساخت vectorstore
print(sample_text[:300], "...")

هوش مصنوعی (به انگلیسی: Artificial intelligence) (سَرنام انگلیسی: AI)، که در برخی منابع علمی هوشواره نیز نامیده می‌شود، هوشی است که به‌دست ماشین‌ها پدید می‌آید، در برابر هوش طبیعی که توسط جانوران شامل انسان‌ها نمایش می‌یابد. ولی پیش از هرچیز باید این موضوع را دانست که کلمه هوش، نشان دهنده امکان استد ...


In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
chunks = splitter.split_text(sample_text)
docs = [Document(page_content=c) for c in chunks]

print(f"تعداد chunk های ایجاد شده: {len(docs)}")

تعداد chunk های ایجاد شده: 8


In [11]:
import faiss
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore

index = faiss.IndexFlatL2(len(hf_embedding.embed_query("سلام")))

vector_store = FAISS(
    embedding_function=hf_embedding,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

vector_store.add_documents(docs)
print("vectorstore با", vector_store.index.ntotal, "بردار ساخته شد.")

vectorstore با 8 بردار ساخته شد.


/var/folders/41/qbs3krkj7yv0sqwyflrs7jxw0000gn/T/ipykernel_60902/3286205869.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


تست جستجوی معنایی روی vectorstore ساخته‌شده:

In [12]:
query = "هوش مصنوعی چه کاربردهایی دارد؟"
results = vector_store.similarity_search(query, k=2)

for r in results:
    print("-", r.page_content[:200], "...")

- سطوح سامانه‌های بازی استراتژیک (همچون شطرنج و گو). با بیشتر شدن توانایی ماشین‌ها، وظایفی که نیازمند «هوشمندی» هستند اغلب از تعریف هوش مصنوعی برداشته می‌شود، پدیده‌ای که به آن اثر هوش مصنوعی گفته می‌شو ...
- و روزمره‌ای شده است. (استفاده از هوش مصنوعی در زمینه‌هایی مانند پزشکی و آموزش رو به افزایش است). ...


### مقایسه با مدل embedding از OpenAI

در این بخش، نتیجه‌ی عملکرد مدل embedding لوکال (ParsBERT) را با مدل `text-embedding-3-large` از OpenAI (از طریق OpenRouter، مطابق پیکربندی موجود در `.env` پروژه) مقایسه می‌کنیم.

In [13]:
import os
import time

from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

load_dotenv("../.env")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

openai_embedding = OpenAIEmbeddings(
    model="text-embedding-3-large",
    api_key=openrouter_api_key,
    base_url="https://openrouter.ai/api/v1",
)

t0 = time.time()
openai_vec = openai_embedding.embed_query(text)
t_openai = time.time() - t0

t0 = time.time()
parsbert_vec = hf_embedding.embed_query(text)
t_parsbert = time.time() - t0

print(f"OpenAI  (text-embedding-3-large, remote via OpenRouter): dim={len(openai_vec):>5}  time={t_openai:.3f}s")
print(f"ParsBERT (bert-base-parsbert-uncased, local/CPU):        dim={len(parsbert_vec):>5}  time={t_parsbert:.3f}s")

OpenAI  (text-embedding-3-large, remote via OpenRouter): dim= 3072  time=1.359s
ParsBERT (bert-base-parsbert-uncased, local/CPU):        dim=  768  time=0.065s


مقایسه‌ی جستجوی معنایی بین دو مدل روی همان مجموعه‌ی chunk‌ها:

In [14]:
openai_index = faiss.IndexFlatL2(len(openai_vec))
openai_store = FAISS(
    embedding_function=openai_embedding,
    index=openai_index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)
openai_store.add_documents(docs)

print("نتیجه‌ی ParsBERT (لوکال):")
for r in vector_store.similarity_search(query, k=1):
    print(" -", r.page_content[:150], "...")

print("\nنتیجه‌ی OpenAI (از طریق OpenRouter):")
for r in openai_store.similarity_search(query, k=1):
    print(" -", r.page_content[:150], "...")

print(
    "\nجمع‌بندی: مدل OpenAI بردار با ابعاد بسیار بزرگ‌تر "
    f"({len(openai_vec)} در مقابل {len(parsbert_vec)}) تولید می‌کند و نیازمند اتصال اینترنت "
    "و پرداخت هزینه بر اساس API است، در حالی که ParsBERT به‌صورت کامل لوکال، رایگان و بدون نیاز "
    "به اینترنت اجرا می‌شود اما چون به صورت اختصاصی برای زبان فارسی fine-tune شده، "
    "می‌تواند برای متون فارسی گزینه‌ی سبک‌تر و مناسبی باشد."
)

نتیجه‌ی ParsBERT (لوکال):
 - سطوح سامانه‌های بازی استراتژیک (همچون شطرنج و گو). با بیشتر شدن توانایی ماشین‌ها، وظایفی که نیازمند «هوشمندی» هستند اغلب از تعریف هوش مصنوعی برداشته م ...

نتیجه‌ی OpenAI (از طریق OpenRouter):


 - هوش مصنوعی (به انگلیسی: Artificial intelligence) (سَرنام انگلیسی: AI)، که در برخی منابع علمی هوشواره نیز نامیده می‌شود، هوشی است که به‌دست ماشین‌ها پد ...

جمع‌بندی: مدل OpenAI بردار با ابعاد بسیار بزرگ‌تر (3072 در مقابل 768) تولید می‌کند و نیازمند اتصال اینترنت و پرداخت هزینه بر اساس API است، در حالی که ParsBERT به‌صورت کامل لوکال، رایگان و بدون نیاز به اینترنت اجرا می‌شود اما چون به صورت اختصاصی برای زبان فارسی fine-tune شده، می‌تواند برای متون فارسی گزینه‌ی سبک‌تر و مناسبی باشد.


---

## بخش ۳: استفاده از قابلیت فیلتر کردن بر اساس متادیتا در جستجوی برداری

در این بخش یاد می‌گیریم چگونه با استفاده از متادیتا، جستجوی برداری را به یک زیرمجموعه‌ی خاص از documentها محدود کنیم.

### بخش اول: لود کردن داده

به‌جای فایل‌های PDF/txt دلخواه، سه صفحه‌ی واقعی از **ویکی‌پدیای فارسی** که با موضوع این بوت‌کمپ (هوش مصنوعی) مرتبط هستند انتخاب و از طریق Wikipedia API دانلود شده‌اند (در پوشه‌ی `data/`):

| منبع | عنوان مقاله | فایل |
|---|---|---|
| `source_01` | هوش مصنوعی | `data/ai.txt` |
| `source_02` | یادگیری ماشین | `data/ml.txt` |
| `source_03` | پردازش زبان‌های طبیعی | `data/nlp.txt` |

هر مقاله را به chunk‌های حدود ۵۰۰ کاراکتری تبدیل کرده و برای هر یک متادیتای `source` متمایز تخصیص می‌دهیم.

In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

sources = {
    "source_01": ("هوش مصنوعی", "data/ai.txt"),
    "source_02": ("یادگیری ماشین", "data/ml.txt"),
    "source_03": ("پردازش زبان‌های طبیعی", "data/nlp.txt"),
}

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

all_documents = []
for source_id, (title, path) in sources.items():
    with open(path, encoding="utf-8") as f:
        raw_text = f.read()

    raw_text = raw_text[:4000]  # برای اجرای سریع‌تر روی CPU، ابتدای هر مقاله استفاده می‌شود
    doc_chunks = splitter.split_text(raw_text)

    for chunk in doc_chunks:
        all_documents.append(
            Document(page_content=chunk, metadata={"source": source_id, "title": title})
        )

    print(f"{source_id} ({title}): {len(doc_chunks)} chunk ایجاد شد")

print("\nمجموع Document های ایجاد شده:", len(all_documents))

source_01 (هوش مصنوعی): 11 chunk ایجاد شد
source_02 (یادگیری ماشین): 13 chunk ایجاد شد
source_03 (پردازش زبان‌های طبیعی): 14 chunk ایجاد شد

مجموع Document های ایجاد شده: 38


### بخش دوم: ایجاد vectorstore و جستجو بر اساس متادیتا

In [16]:
index_3 = faiss.IndexFlatL2(len(hf_embedding.embed_query("سلام")))

vector_store_3 = FAISS(
    embedding_function=hf_embedding,
    index=index_3,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

vector_store_3.add_documents(all_documents)
print("vectorstore با", vector_store_3.index.ntotal, "بردار از هر ۳ منبع ساخته شد.")

vectorstore با 38 بردار از هر ۳ منبع ساخته شد.


جستجو با محدود کردن نتایج فقط به `source_01` (هوش مصنوعی):

In [17]:
results = vector_store_3.similarity_search(
    "کاربردهای هوش مصنوعی در زندگی روزمره چیست؟",
    k=2,
    filter={"source": "source_01"},
)

print(results)
for res in results:
    print(f"* {res.page_content} [{res.metadata}]")

[Document(id='43bc6802-0ff1-4d38-bb17-3b458a250300', metadata={'source': 'source_01', 'title': 'هوش مصنوعی'}, page_content='نیازمند «هوشمندی» هستند اغلب از تعریف هوش مصنوعی برداشته می\u200cشود، پدیده\u200cای که به آن اثر هوش مصنوعی گفته می\u200cشود. به عنوان مثال، فهم نوری کاراکتر را اغلب از چیزهایی که هوش مصنوعی در نظر گرفته می\u200cشوند مستثنی می\u200cکنند، چرا که این فناوری تبدیل به فناوری عادی و روزمره\u200cای شده است. (استفاده از هوش مصنوعی در زمینه\u200cهایی مانند پزشکی و آموزش رو به افزایش است).'), Document(id='3812d081-98fc-42a8-ab41-f5df8bf604d1', metadata={'source': 'source_01', 'title': 'هوش مصنوعی'}, page_content='کتاب\u200cهای هوش مصنوعی پیشرو، این شاخه را به عنوان شاخه مطالعه بر روی «عوامل هوشمند» تعریف می\u200cکنند: هر سامانه\u200cای که محیط خود را درک کرده و کنش\u200cهایی را انجام می\u200cدهد که شانسش را در دستیابی به اهدافش بیشینه می\u200cسازد. برخی از منابع شناخته شده از اصطلاح «هوش مصنوعی» جهت توصیف ماشینی استفاده می\u200cکنند که عملکردهای «شناختی» را از روی ذهن انسا

جستجو با محدود کردن نتایج فقط به `source_02` (یادگیری ماشین):

In [18]:
results_ml = vector_store_3.similarity_search(
    "یادگیری با نظارت (supervised learning) چگونه کار می‌کند؟",
    k=2,
    filter={"source": "source_02"},
)

for res in results_ml:
    print(f"* {res.page_content} [{res.metadata}]")

* یادگیری ماشین (به انگلیسی: Machine learning) یا اِم‌اِل (کوته‌نوشت: ML)، مطالعه الگوریتم‌ها و مدل‌های آماری مورد استفاده سیستم‌های کامپیوتری است که به‌جای استفاده از دستورالعمل‌های واضح، از الگوها و استنباط برای انجام وظایف استفاده می‌کنند. یادگیری ماشینی علمی است که باعث می‌شود رایانه‌ها بدون نیاز به یک برنامه صریح در مورد یک موضوع خاص یاد بگیرند. به عنوان زیر مجموعه‌ای از هوش مصنوعی، الگوریتم‌های یادگیری ماشینی یک مدل ریاضی بر اساس داده‌های نمونه یا داده‌های آموزش به منظور پیش‌بینی یا [{'source': 'source_02', 'title': 'یادگیری ماشین'}]
* == هدف‌ها و انگیزه‌ها ==
هدف یادگیری ماشینی این است که رایانه‌ها و سامانه‌ها بتوانند به تدریج و با افزایش داده‌ها کارایی بهتری در انجام وظیفه مورد نظر پیدا کند.
گستره این وظیفه می‌تواند از تشخیص خودکار چهره با دیدن چند نمونه از چهره مورد نظر تا فراگیری شیوه گام‌برداری روبات‌های دوپا با دریافت سیگنال پاداش و تنبیه باشد.
طیف پژوهش‌هایی که در یادگیری ماشینی می‌شود گسترده است. [{'source': 'source_02', 'title': 'یادگیری ماشین'}]


مقایسه‌ی جستجوی **بدون فیلتر** (روی هر سه منبع) در برابر جستجوی **فیلترشده** (فقط `source_03`، پردازش زبان طبیعی) برای یک کوئری مشترک:

In [19]:
query = "رابطه‌ی پردازش زبان طبیعی و یادگیری ماشین چیست؟"

print("بدون فیلتر (جستجو روی هر ۳ منبع):")
unfiltered = vector_store_3.similarity_search(query, k=5)
for r in unfiltered:
    print(" -", r.metadata["source"], f"({r.metadata['title']}):", r.page_content[:80], "...")

print("\nفقط source_03 (پردازش زبان‌های طبیعی):")
filtered = vector_store_3.similarity_search(query, k=5, filter={"source": "source_03"})
for r in filtered:
    print(" -", r.metadata["source"], f"({r.metadata['title']}):", r.page_content[:80], "...")

بدون فیلتر (جستجو روی هر ۳ منبع):
 - source_03 (پردازش زبان‌های طبیعی): پردازش زبان‌های طبیعی یکی از زیرشاخه‌های مهم در حوزهٔ علوم رایانه، هوش مصنوعی اس ...
 - source_02 (یادگیری ماشین): رابطه بین هوش مصنوعی و یادگیری ماشین را می‌توانیم اینطور بیان کنیم که: هوش مصنوع ...
 - source_02 (یادگیری ماشین): یادگیری ماشین (به انگلیسی: Machine learning) یا اِم‌اِل (کوته‌نوشت: ML)، مطالعه  ...
 - source_02 (یادگیری ماشین): یادگیری ماشینی کمک فراوانی به صرفه جویی در هزینه‌های عملیاتی و بهبود سرعت عمل تج ...
 - source_01 (هوش مصنوعی): کاربردهای هوش مصنوعی شامل موتورهای جستجو پیشرفتهٔ وب (مثل گوگل و بینگ)، سامانهٔ  ...

فقط source_03 (پردازش زبان‌های طبیعی):
 - source_03 (پردازش زبان‌های طبیعی): پردازش زبان‌های طبیعی یکی از زیرشاخه‌های مهم در حوزهٔ علوم رایانه، هوش مصنوعی اس ...
 - source_03 (پردازش زبان‌های طبیعی): هدف اصلی در پردازش زبان طبیعی، ساخت تئوری‌هایی محاسباتی از زبان، با استفاده از ا ...
 - source_03 (پردازش زبان‌های طبیعی): نمونه‌هایی از کاربردهای گفتاری پردازش زبان عبارتند از: سیستم‌ه

---

## جمع‌بندی

در این notebook هر سه بخش تمرین به‌صورت کامل و لوکال (بدون نیاز به Google Colab) پیاده‌سازی شد:

1. **بخش ۱:** اتصال به مدل زبانی `llama3.1:8b` سرو شده روی Ollama محلی، از طریق `langchain-ollama` و تست با پرامپت‌های فارسی و انگلیسی.
2. **بخش ۲:** دانلود و اجرای مدل embedding فارسی `ParsBERT` به‌صورت لوکال، ساخت یک vectorstore با FAISS، جستجوی معنایی، و مقایسه با مدل `text-embedding-3-large` از OpenAI.
3. **بخش ۳:** ساخت vectorstore از سه مقاله‌ی واقعی ویکی‌پدیای فارسی با متادیتای متمایز برای هرکدام، و نمایش قابلیت فیلتر کردن جستجوی برداری بر اساس متادیتا.